# Bolo 02: Whisper fine-tune for Roman-script command Hinglish

Base: **Oriserve/Whisper-Hindi2Hinglish-Swift** (whisper-base, 72.6M, Apache-2.0, Roman output).
Targets: Bolo's canonical Roman `asr_text`. Training audio: the synthetic speech from notebook 01, with
on-the-fly noise/reverb/speed/phone-band augmentation (`bolo.audio.augment`).

**Before running**
1. Kaggle *Add input*: the **bolo-bundle** dataset and the **output of notebook 01** (bolo-tts).
2. Settings: **GPU T4 x2**, **Internet on** (to download base models, ~0.3-1 GB).
3. Save Version -> Save & Run All. Roughly: baselines ~10 min, fine-tune ~20-40 min (estimate; check the log).

**Honesty notes**
- Every number here is on *synthetic* speech (held-out voices, held-out templates/values). The real
  benchmark is the human `recordings/` set, evaluated locally in Phase 5 (audio never leaves the laptop).
- WER is reported raw, and also `phonetic_wer`, which forgives Roman spelling variants (kholo/kolo).
  Devanagari output (plain Whisper, language=hi) is transliterated before scoring, which still penalises
  English loanwords (टर्मिनल -> "tarminal" != "terminal"). That is a real cost of Devanagari output for our use.

**Output** `/kaggle/working/bolo-whisper/` (~290 MB safetensors + processor) and `results_asr.json`.
Download both into `models/bolo-whisper/` and `results/asr/` locally (models/ is gitignored).

In [ ]:
import glob, os, sys, json
# Kaggle: bundle uploaded as a private dataset -> /kaggle/input/...; Colab: unzip bolo-bundle.zip into /content
cands = glob.glob("/kaggle/input/**/bolo-bundle/src/bolo/__init__.py", recursive=True) + \
        glob.glob("/content/**/bolo-bundle/src/bolo/__init__.py", recursive=True)
assert cands, "bolo-bundle not found: add the bolo-bundle dataset (Kaggle) or unzip it in /content (Colab)"
BUNDLE = cands[0].split("/src/bolo/")[0]
sys.path.insert(0, BUNDLE + "/src")
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
print("bundle:", BUNDLE, "| work dir:", WORK)

tm = (glob.glob("/kaggle/input/**/bolo-tts-v1/**/manifest.jsonl", recursive=True)
      + glob.glob("/kaggle/input/**/bolo-tts/manifest.jsonl", recursive=True) + glob.glob(WORK + "/bolo-tts/manifest.jsonl"))
assert tm, "add the bolo-tts-v1 dataset (or notebook 01's output) as an input"
TTS_MAN = tm[0]
print("tts manifest:", TTS_MAN)

### Real rooms and real noise (OpenSLR 28, Apache-2.0)
Measured room impulse responses + point-source noises (Ko et al., 2017). Downloaded here on Kaggle, never
committed. Used by `augment(..., rirs=...)` for 70% of reverberated clips, and as noise for SNR mixing and for
noise-only clips with an empty transcript (teaches the model that "nothing said" means no words).

In [ ]:
import os, glob, subprocess, numpy as np
from bolo.audio.augment import resample
RIR_DIR = WORK + "/rirs"
if not os.path.exists(RIR_DIR + "/RIRS_NOISES"):
    os.makedirs(RIR_DIR, exist_ok=True)
    subprocess.run(["wget", "-q", "-O", RIR_DIR + "/rirs.zip", "https://www.openslr.org/resources/28/rirs_noises.zip"], check=True)
    subprocess.run(["unzip", "-q", RIR_DIR + "/rirs.zip", "-d", RIR_DIR], check=True)
    os.remove(RIR_DIR + "/rirs.zip")

import soundfile as sf
def _load(p, max_s):
    x, sr = sf.read(p, dtype="float32", always_2d=True)
    x = x[:, 0]  # one microphone of the array
    x = resample(x.astype("float32"), sr, 16000)[: int(16000 * max_s)]
    return x / (np.abs(x).max() + 1e-6)

rng0 = np.random.default_rng(0)
rir_files = sorted(glob.glob(RIR_DIR + "/RIRS_NOISES/real_rirs_isotropic_noises/*rir*.wav"))
noise_files = sorted(glob.glob(RIR_DIR + "/RIRS_NOISES/pointsource_noises/*.wav"))
RIRS = [_load(p, 1.0) for p in rir_files]
NOISES = [_load(p, 8.0) for p in rng0.choice(noise_files, size=min(400, len(noise_files)), replace=False)]
print(len(RIRS), "real RIRs,", len(NOISES), "noises")

In [ ]:
import torch, time
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from bolo.asr import whisper_ft as W
from bolo.config import load_config
from bolo.normalize import Normalizer
nz = Normalizer.from_config(load_config())
rows = W.load_manifest(TTS_MAN, for_training=True)
split = {s: [r for r in rows if r["split"] == s] for s in ("train", "dev", "test")}
split["train"] = split["train"] + W.nonspeech_rows(max(1, len(split["train"]) * 3 // 100))  # ~3% noise-only
print({k: len(v) for k, v in split.items()})
DEV = "cuda"
RESULTS = {"data": {k: len(v) for k, v in split.items()}, "baselines": {}, "finetuned": {}}

## Baselines on the synthetic test set (held-out voices)

In [ ]:
BASELINES = [("openai/whisper-base", "hi"), ("openai/whisper-base", "en"), ("openai/whisper-small", "hi"),
             ("Oriserve/Whisper-Hindi2Hinglish-Swift", "en"), ("Oriserve/Whisper-Hindi2Hinglish-Swift", "hi")]
hyps_all = {}
for mid, lang in BASELINES:
    proc = WhisperProcessor.from_pretrained(mid)
    m = WhisperForConditionalGeneration.from_pretrained(mid, torch_dtype=torch.float16).to(DEV)
    s, h = W.evaluate(m, proc, split["test"], nz, language=lang, device=DEV)
    RESULTS["baselines"][f"{mid}|{lang}"] = s
    hyps_all[f"{mid}|{lang}"] = h
    print(mid, lang, s)
    del m; torch.cuda.empty_cache()

## Fine-tune

In [ ]:
CFG = dict(epochs=5, batch_size=32, lr=1e-5, warmup=50, eval_every=150, seed=0, freeze_encoder=False)
proc = WhisperProcessor.from_pretrained(W.BASE)
model = WhisperForConditionalGeneration.from_pretrained(W.BASE)
OUTDIR = WORK + "/bolo-whisper"
t0 = time.time()
res = W.train(model, proc, split["train"], split["dev"], OUTDIR, device=DEV, normalizer=nz, noises=NOISES, rirs=RIRS, **CFG)
RESULTS["train"] = {"config": CFG, "base": W.BASE, "real_rirs": len(RIRS), "noises": len(NOISES),
                    "nonspeech_rows": sum(bool(r.get("nonspeech")) for r in split["train"]), "minutes": round((time.time() - t0) / 60, 1), **res}

In [ ]:
best = WhisperForConditionalGeneration.from_pretrained(OUTDIR, torch_dtype=torch.float16).to(DEV)
bproc = WhisperProcessor.from_pretrained(OUTDIR)
s, h = W.evaluate(best, bproc, split["test"], nz, language="en", device=DEV)
RESULTS["finetuned"]["bolo-whisper|en"] = s
hyps_all["bolo-whisper|en"] = h
print(s)
# per-utterance outputs for error analysis (synthetic text only; no audio)
RESULTS["test_utts"] = [{"utt_id": r["utt_id"], "ref": W.target(r), "speaker": r["speaker"],
                         **{k: v[i] for k, v in hyps_all.items()}} for i, r in enumerate(split["test"])]
json.dump(RESULTS, open(WORK + "/results_asr.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps({k: v for k, v in RESULTS.items() if k != "test_utts"}, indent=1)[:3000])

Sample errors from the fine-tuned model:

In [ ]:
bad = [u for u in RESULTS["test_utts"] if u["bolo-whisper|en"].lower().strip(" .?!") != u["ref"]]
for u in bad[:25]:
    print(f'{u["ref"]!r:45} -> {u["bolo-whisper|en"]!r}')